# Line-settings checker

A quick way to test the emission-line settings for a new dataset **before** running the full analysis
(`02_template_new_galaxy.ipynb`). For every line it:

1. shows the **wavelength window**: summed spectrum of the brightest spaxels, the line region, the continuum fit,
   and any neighbouring catalogue lines that could blend with it;
2. fits the single and double Gaussians in **a few spaxels only** (the brightest plus random ones), so it runs in
   seconds, and plots them;
3. prints a **summary** flagging failed fits, fits at a parameter bound, and spaxels where the two Gaussians come
   out with similar widths.

Adjust `LINE_SETTINGS`, re-run, and when everything looks right copy `LINE_SETTINGS` (and `FIT_OPTIONS`) into notebook 02.

## Settings - EDIT

In [ ]:
GALAXY       = "MyGalaxy"
DATA_DIR     = "path/to/cubes"
RESULTS_DIR  = "results"
Z            = None              # redshift  <-- REQUIRED
DISTANCE_MPC = None

N_SPAXELS = 6                    # spaxels fitted per line (brightest + random)
SEED      = 0                    # change to draw other random spaxels

SAVE_FIGURES = "none"            # "png", "pdf", "both" or "none"
FIGURE_DPI   = 150

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
try:
    import breaking_the_blend as btb
except ModuleNotFoundError:   # not installed (pip install -e .): use the copy in this repository
    import sys, os
    sys.path.insert(0, os.path.abspath(".."))
    import breaking_the_blend as btb
from breaking_the_blend import plotting as bplt     # also sets the publication plotting style

if Z is None:
    raise ValueError("Set the redshift Z")
proj = btb.Project(f"{GALAXY}_line_check", data_dir=DATA_DIR, z=Z, distance_mpc=DISTANCE_MPC,
                   results_dir=RESULTS_DIR, save_figures=SAVE_FIGURES, figure_dpi=FIGURE_DPI)
files = proj.find_cubes(exclude=["*bkg*"])
cubes = proj.load_cubes(agn_overrides={})
fig = bplt.plot_cubes_overview(cubes);  proj.savefig(fig, "cubes_overview")

## Lines and settings - EDIT

Same format as notebook 02. Settings: `cube`, `line_region` (rest $\mu$m), `fit_window`, `mask_percentile`,
`p0_amp_frac`, `p0_dv_kms`, `p0_sigma_kms` (initial narrow / broad values), `sigma_bounds_kms`, `vel_bounds_kms`.

In [ ]:
LINES = ["FeII", "ArII", "NeII", "SIII", "ArIII", "SIV", "NeIII", "OIV", "ArVI", "MgIV", "NeV", "NeVI"]

COMMON_SETTINGS = dict(mask_percentile=40)
LINE_SETTINGS = {
    "NeV": dict(line_region=(14.295, 14.345), fit_window=(14.265, 14.350)),   # keep [Cl II] out
    # "NeII": dict(line_region=(12.79, 12.84), p0_sigma_kms=(90, 200)),
}
FIT_OPTIONS = dict(fitter="trf", weighting="none", p0_center="moment", percentile_over="fov", dilate=2)

lines = btb.get_lines(LINES)
lines = {n: l.update(**{**COMMON_SETTINGS, **LINE_SETTINGS.get(n, {})}) for n, l in lines.items()}
lines = btb.assign_cubes(lines, cubes, drop_missing=True)

## 1. Wavelength windows

Check that the **shaded line region** contains the whole line (including broad wings) and nothing else, that the
**continuum fit** (blue) follows the line-free spectrum on both sides, and that no **neighbouring line** (purple, dotted)
falls inside the window. If one does, narrow `line_region` / `fit_window` to exclude it.

In [ ]:
names = list(lines)
ncols = 3
fig, axes = plt.subplots(int(np.ceil(len(names) / ncols)), ncols, figsize=(6.2 * ncols, 3.9 * np.ceil(len(names) / ncols)),
                         squeeze=False, constrained_layout=True)
for ax in axes.flat[len(names):]:
    ax.axis("off")
for ax, n in zip(axes.flat, names):
    bplt.plot_line_window(cubes[lines[n].cube], lines[n], ax=ax)
proj.savefig(fig, "line_windows")

## 2. Fits in a few spaxels

Only `N_SPAXELS` spaxels per line are fitted. In each panel the narrow (purple) and broad (green) Gaussians should
together follow the data (grey dashed = their sum). Where a spaxel holds only one component the two Gaussians can
come out nearly identical - this is expected and not a problem.

In [ ]:
checks = {n: btb.check_line(cubes[l.cube], l, n_spaxels=N_SPAXELS, seed=SEED, **FIT_OPTIONS)
          for n, l in lines.items()}
summary = btb.check_summary(checks)

In [ ]:
for n, lm in checks.items():
    fig = bplt.plot_spaxel_fits(lm, n=N_SPAXELS)
    proj.savefig(fig, f"spaxel_fits_{n}")

### If something looks wrong

| Symptom | Try |
|---|---|
| Line cut off, or wings outside the shaded region | widen `line_region` |
| Another line inside the window | narrow `line_region` / `fit_window` around the line |
| Continuum fit poor (curved, too high/low) | change `fit_window` (more line-free channels) or `cont_order` |
| Fits failed or at a bound | change `p0_sigma_kms` / `p0_dv_kms`, or widen `sigma_bounds_kms` / `vel_bounds_kms` |
| Narrow and broad similar in most spaxels | the line may have a single component in these data - fine for the outflow fits, but consider leaving it out of the decomposition |
| Too few spaxels kept / too noisy | lower / raise `mask_percentile` |

Change `SEED` to look at other spaxels.

## 3. Settings to copy into notebook 02

In [ ]:
import pprint
print("COMMON_SETTINGS =", COMMON_SETTINGS)
print("FIT_OPTIONS =", FIT_OPTIONS)
print("LINE_SETTINGS = ", end="")
pprint.pprint(LINE_SETTINGS, sort_dicts=False)